# Future-method footprints: a month at a glance

Plots one month file written by `name_satellite.future.processing` (layout:
`docs/future/file-format.md`; method: `docs/future/scoping.md`). Each sounding has a **column
footprint** at three levels, a 0.1° box (±5°), a 0.5° box (±25°) and the global 2.5° grid,
and a histogram of **where its particles were** at the end of the 10-day back-run.

Reading is done by `name_satellite.future.footprint_file`; this notebook only plots.

**Running it.** From the repository: `uv sync --group notebooks`, then either
`uv run --group notebooks --with jupyterlab jupyter lab`, or register the kernel for the JASMIN
Notebook Service with `uv run --group notebooks python -m ipykernel install --user --name name-satellite`.
Set `FILE` and `SOUNDING_NEAR` below and run all cells.

In [ ]:
from pathlib import Path

FILE = Path(
    "/gws/ssde/j25b/acrg/mattrigby/name-satellite-future/"
    "GOSAT-BRAZIL_GOSAT-1_v10.0_201301_10day-2026-11-26/footprints/"
    "GOSAT-BRAZIL_GOSAT-1_v10.0_201301_future.nc"
)
SOUNDING_NEAR = (-60.0, -3.1)  # (lon, lat): the example sounding is the one nearest this (Manaus)
REGION = (-95, -25, -60, 15)  # map extent for the regional plots: west, east, south, north

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.patches import Rectangle

from name_satellite.future.footprint_file import (
    aggregate,
    box_edges,
    finest_footprint,
    footprint_density,
    open_footprints,
    sounding_footprint,
    sounding_table,
)

### Plot style
One look for every figure. Each kind of quantity has its own single-hue colour ramp (light =
small, dark = large), always on a log scale for footprints: they span many orders of magnitude.

In [ ]:
INK, MUTED, FAINT = "#1f2328", "#5f6670", "#d6d9de"
ACCENT = "#d9480f"  # the sounding's position
FINE_COLOUR, MEDIUM_COLOUR = "#5b45b0", "#c2700a"  # the two boxes, where they are compared

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "figure.facecolor": "white",
    "font.size": 9.5, "axes.titlesize": 10.5, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlepad": 8,
    "axes.labelcolor": MUTED, "axes.edgecolor": FAINT, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.major.size": 3, "ytick.major.size": 3,
    "legend.frameon": False, "text.color": INK,
})  # fmt: skip


def ramp(name, colours):
    """A single-hue sequential colour map; values below its range are left transparent."""
    cmap = LinearSegmentedColormap.from_list(name, colours)
    cmap.set_under((1, 1, 1, 0))
    cmap.set_bad((1, 1, 1, 0))
    return cmap


FOOTPRINT_CMAP = ramp(
    "footprint", ["#efedf8", "#c9c1ea", "#9d8fd6", "#6f5cbd", "#46339a", "#21125f"]
)
ENDS_CMAP = ramp("ends", ["#e8f4f2", "#b4dcd5", "#73bdb1", "#35978a", "#146c62", "#05403a"])
DATE_CMAP = ramp("date", ["#cfe0f1", "#8fb5dd", "#4f88c4", "#245c9e", "#0c3466"])

LAND, OCEAN = "#f3f2ef", "#ffffff"


def base_map(ax, extent=None, borders=True):
    """Light land, thin coastlines (and borders), a clean frame."""
    if extent is None:
        ax.set_global()
    else:
        ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor=LAND, edgecolor="none", zorder=0)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), facecolor=OCEAN, edgecolor="none", zorder=0)
    ax.coastlines("50m", linewidth=0.45, color="#7d848d", zorder=3)
    if borders:
        ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.3, edgecolor="#a9afb6",
                       zorder=3)  # fmt: skip
    ax.spines["geo"].set_edgecolor(FAINT)
    return ax


def gridlines(ax, step=10):
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color=FAINT, zorder=1,
                      xlocs=np.arange(-180, 181, step), ylocs=np.arange(-90, 91, step))  # fmt: skip
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = gl.ylabel_style = {"color": MUTED, "size": 8}
    return gl


def draw_box(ax, box, label=None, style="-"):
    """Outline a box (a ``grids.Box``) with an optional label in its top-left corner."""
    ax.add_patch(Rectangle((box.west, box.south), box.east - box.west, box.north - box.south,
                           fill=False, edgecolor=INK, linewidth=0.9, linestyle=style,
                           transform=ccrs.PlateCarree(), zorder=5))  # fmt: skip
    if label:
        ax.text(box.west + 0.6, box.north - 0.6, label, transform=ccrs.PlateCarree(),
                ha="left", va="top", fontsize=8, color=INK, zorder=6,
                bbox={"boxstyle": "round,pad=0.2", "facecolor": "white", "edgecolor": "none",
                      "alpha": 0.85})  # fmt: skip


def mesh(ax, field, cmap, norm):
    """pcolormesh of a (lat, lon) DataArray of cell centres, on a map."""
    return ax.pcolormesh(field["lon"], field["lat"], field.where(field > 0), cmap=cmap, norm=norm,
                         transform=ccrs.PlateCarree(), shading="nearest", rasterized=True,
                         zorder=2)  # fmt: skip


def colourbar(fig, artist, axes, label, **kwargs):
    cb = fig.colorbar(artist, ax=axes, orientation="horizontal", shrink=0.45, aspect=35,
                      pad=0.06, extend="min", **kwargs)  # fmt: skip
    cb.set_label(label, color=MUTED)
    cb.outline.set_visible(False)
    cb.ax.tick_params(colors=MUTED, labelsize=8)
    return cb


def log_norm(values, decades=3.5):
    """Log scale from the 99.9th percentile of the positive values down ``decades`` decades."""
    positive = np.asarray(values)[np.asarray(values) > 0]
    top = np.percentile(positive, 99.9)
    return LogNorm(vmin=top / 10**decades, vmax=top)


PPB = 1e9  # mol/mol -> ppb
DENSITY_LABEL = "Column enhancement from a 1 mol s⁻¹ source in the cell (ppb)"

## The file

In [ ]:
ds = open_footprints(FILE)
table = sounding_table(ds)
month = pd.Timestamp(table["time"].min()).strftime("%B %Y")
print(
    f"{FILE.name}: {len(table):,} soundings, {table['release_group'].nunique():,} release groups,"
    f" {table['time'].dt.normalize().nunique()} days, {month}"
)
print(
    f"Settings: {ds.attrs.get('options')}; max travel time {ds.attrs.get('max_age_hours')} h;"
    f" made by commit {ds.attrs.get('software_commit')} on {ds.attrs.get('created')}"
)
table.head()

## The month's soundings
Each dot is a sounding, coloured by its day; soundings in the same 0.1° cell within a short
time share one release (D20).

In [ ]:
fig = plt.figure(figsize=(11, 5.2))
grid = fig.add_gridspec(1, 2, width_ratios=[1.15, 1], wspace=0.12)
ax = base_map(fig.add_subplot(grid[0], projection=ccrs.PlateCarree()), REGION)
gridlines(ax)
day = table["time"].dt.day
dots = ax.scatter(
    table["lon"],
    table["lat"],
    c=day,
    cmap=DATE_CMAP,
    vmin=1,
    vmax=31,
    s=16,
    edgecolors="white",
    linewidths=0.4,
    transform=ccrs.PlateCarree(),
    zorder=4,
)
ax.set_title(f"{len(table):,} soundings, {month}")
cb = colourbar(fig, dots, ax, "Day of the month", ticks=[1, 8, 15, 22, 29])
cb.ax.xaxis.set_tick_params(which="minor", size=0)

ax = fig.add_subplot(grid[1])
per_day = day.value_counts().reindex(range(1, 32), fill_value=0)
ax.bar(
    per_day.index, per_day.values, width=0.75, color=DATE_CMAP(0.65), edgecolor="white", linewidth=1
)
ax.set_xlim(0.3, 31.7)
ax.set_xticks([1, 8, 15, 22, 29])
ax.set_xlabel("Day of the month")
ax.set_title("Soundings per day")
ax.grid(axis="y", color=FAINT, linewidth=0.5)
ax.set_axisbelow(True)
busiest = per_day.idxmax()
ax.annotate(
    f"{per_day.max()} on the {busiest}th",
    (busiest, per_day.max()),
    xytext=(0, 4),
    textcoords="offset points",
    ha="center",
    fontsize=8,
    color=MUTED,
)
ax.set_box_aspect(0.8)
plt.show()

## One sounding on its three levels
The example sounding is the one nearest `SOUNDING_NEAR`. Each level is drawn where it exists,
finest on top, as a **density** (footprint ÷ cell area: the enhancement from a point source of
1 mol s⁻¹ in the cell), so the three levels share one colour scale and meet without a jump
where they agree.

In [ ]:
distance = np.hypot(table["lon"] - SOUNDING_NEAR[0], table["lat"] - SOUNDING_NEAR[1])
SOUNDING = distance.idxmin()
row = table.loc[SOUNDING]
levels = {name: footprint_density(sounding_footprint(ds, SOUNDING, name)) * PPB
          for name in ("coarse", "medium", "fine")}  # fmt: skip
boxes = {name: box_edges(ds, SOUNDING, name) for name in ("medium", "fine")}
norm = log_norm(np.concatenate([levels[name].values.ravel() for name in levels]), decades=4)
print(
    f"Sounding {SOUNDING}: {row['time']:%d %b %Y %H:%M} UTC"
    f" at {row['lat']:.2f}°, {row['lon']:.2f}°;"
    f" {row['in_fine_box']:.0%} of its response in the 0.1° box, {row['in_medium_box']:.0%} in"
    f" the 0.5° box"
)

fig = plt.figure(figsize=(11, 10.5))
grid = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=0.14, wspace=0.12)
panels = [
    (
        "Global: 2.5° cells, with the 0.5° and 0.1° boxes on top",
        ccrs.Robinson(central_longitude=-40),
        None,
        grid[0, :],
    ),
    ("±25° box: 0.5° cells", ccrs.PlateCarree(), boxes["medium"], grid[1, 0]),
    ("±5° box: 0.1° cells", ccrs.PlateCarree(), boxes["fine"], grid[1, 1]),
]
for title, projection, extent_box, place in panels:
    ax = fig.add_subplot(place, projection=projection)
    extent = None if extent_box is None else (extent_box.west, extent_box.east,
                                              extent_box.south, extent_box.north)  # fmt: skip
    base_map(ax, extent, borders=extent_box is not None)
    for name in ("coarse", "medium", "fine"):
        artist = mesh(ax, levels[name], FOOTPRINT_CMAP, norm)
    if extent_box is None:
        draw_box(ax, boxes["medium"])
        draw_box(ax, boxes["fine"])
    else:
        gridlines(ax, step=10 if extent_box is boxes["medium"] else 2.5)
        if extent_box is boxes["medium"]:
            draw_box(ax, boxes["fine"], "0.1° box")
    ax.plot(
        row["lon"],
        row["lat"],
        marker="*",
        markersize=11,
        color=ACCENT,
        markeredgecolor="white",
        markeredgewidth=0.7,
        transform=ccrs.PlateCarree(),
        zorder=7,
    )
    ax.set_title(title)
fig.suptitle(
    f"Sounding {SOUNDING}, {row['time']:%d %b %Y %H:%M} UTC",
    x=0.125,
    ha="left",
    y=0.93,
    fontsize=12,
    fontweight="bold",
)
colourbar(fig, artist, fig.axes, DENSITY_LABEL)
plt.show()

## Where its particles were after 10 days
`end_positions` gives the column-weighted fraction of the sounding's particles in each
2.5° × 1 km bin when they reached the maximum travel time, plus those that left through the
20 km top earlier. An inversion samples a global CH₄ field there (the boundary term).

In [ ]:
ends = ds["end_positions"].sel(sounding_id=SOUNDING).load() / row["column_weight"]
in_column = ends.isel(end_height=slice(0, -1))
column_map = in_column.sum("end_height").rename(coarse_lat="lat", coarse_lon="lon")
profile = in_column.sum(("coarse_lat", "coarse_lon"))
left_top = float(ends.isel(end_height=-1).sum())

fig = plt.figure(figsize=(12, 4.6))
grid = fig.add_gridspec(1, 2, width_ratios=[2.4, 1], wspace=0.12)
ax = base_map(fig.add_subplot(grid[0], projection=ccrs.Robinson(central_longitude=-40)))
artist = mesh(ax, column_map * 100, ENDS_CMAP, log_norm(column_map.values * 100, decades=3))
ax.plot(
    row["lon"],
    row["lat"],
    marker="*",
    markersize=11,
    color=ACCENT,
    markeredgecolor="white",
    markeredgewidth=0.7,
    transform=ccrs.PlateCarree(),
    zorder=7,
)
ax.set_title("Share of the column in each 2.5° column (0–20 km)")
colourbar(fig, artist, ax, "% of the column's particles")

ax = fig.add_subplot(grid[1])
heights_km = profile["end_height"].values / 1000
ax.barh(
    heights_km,
    profile.values * 100,
    height=0.8,
    color=ENDS_CMAP(0.6),
    edgecolor="white",
    linewidth=0.8,
)
ax.set_ylim(0, 20)
ax.set_ylabel("Height above ground (km)")
ax.set_xlabel("% of the column's particles")
ax.set_title("By height")
ax.grid(axis="x", color=FAINT, linewidth=0.5)
ax.set_axisbelow(True)
ax.text(
    0.97,
    0.97,
    f"{left_top:.1%} left through\nthe 20 km top",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=8.5,
    color=MUTED,
)
plt.show()

## The whole month
Every sounding's footprint summed on the 0.5° global grid, each taking its finest level
(`finest_footprint`: 0.5° inside its own ±25° box, its 2.5° cells spread evenly elsewhere),
shown as a density. The regional view shows where the month's soundings, together, are
sensitive to surface fluxes; the global view shows how far 10 days reach.

In [ ]:
total = footprint_density(finest_footprint(ds, "medium")) * PPB
fig = plt.figure(figsize=(13, 5.6))
grid = fig.add_gridspec(1, 2, width_ratios=[1, 1.45], wspace=0.06)
norm = log_norm(total.values, decades=3.5)
ax = base_map(fig.add_subplot(grid[0], projection=ccrs.PlateCarree()), REGION)
gridlines(ax)
artist = mesh(ax, total, FOOTPRINT_CMAP, norm)
ax.scatter(
    table["lon"],
    table["lat"],
    s=1.5,
    color=ACCENT,
    alpha=0.5,
    linewidths=0,
    transform=ccrs.PlateCarree(),
    zorder=4,
)
ax.set_title("South America, 0.5°")
ax = base_map(fig.add_subplot(grid[1], projection=ccrs.Robinson(central_longitude=-40)))
mesh(ax, total, FOOTPRINT_CMAP, norm)
ax.set_title("Globe")
fig.suptitle(
    f"Sum of all {len(table):,} footprints, {month}",
    x=0.125,
    ha="left",
    y=0.98,
    fontsize=12,
    fontweight="bold",
)
colourbar(fig, artist, fig.axes, "Summed enhancement from a 1 mol s⁻¹ source in the cell (ppb)")
plt.show()

## Where the month's particles were after 10 days
The mean over soundings of the end-position fractions: by 2.5° column, and by latitude and
height (summed over longitude).

In [ ]:
weights = ds["column_weight"].sum("release_level")
mean_ends = (ds["end_positions"] / weights).mean("sounding_id").load()
in_column = mean_ends.isel(end_height=slice(0, -1))
column_map = in_column.sum("end_height").rename(coarse_lat="lat", coarse_lon="lon") * 100
section = in_column.sum("coarse_lon") * 100

fig = plt.figure(figsize=(13, 4.8))
grid = fig.add_gridspec(1, 2, width_ratios=[1.5, 1], wspace=0.12)
ax = base_map(fig.add_subplot(grid[0], projection=ccrs.Robinson(central_longitude=-40)))
artist = mesh(ax, column_map, ENDS_CMAP, log_norm(column_map.values, decades=3))
ax.set_title("By 2.5° column (0–20 km)")
colourbar(fig, artist, ax, "% of particles, mean over soundings")

ax = fig.add_subplot(grid[1])
image = ax.pcolormesh(
    section["coarse_lat"],
    section["end_height"] / 1000,
    section.where(section > 0),
    cmap=ENDS_CMAP,
    norm=log_norm(section.values, decades=3),
    shading="nearest",
    rasterized=True,
)
ax.set_xlim(-90, 90)
ax.set_xticks([-90, -60, -30, 0, 30, 60, 90])
ax.set_xticklabels(["90°S", "60°S", "30°S", "0°", "30°N", "60°N", "90°N"])
ax.set_ylim(0, 20)
ax.set_ylabel("Height above ground (km)")
top = float((mean_ends.isel(end_height=-1).sum()) * 100)
ax.set_title(f"By latitude and height\n{top:.2f}% left through the 20 km top", linespacing=1.5)
ax.spines[["left", "bottom"]].set_visible(False)
cb = colourbar(fig, image, ax, "% of particles")
fig.suptitle(
    f"Particle positions at the end of the 10-day back-runs, {month}",
    x=0.125,
    ha="left",
    y=1.0,
    fontsize=12,
    fontweight="bold",
)
plt.show()

## How much of each footprint lies in the boxes
The share of each sounding's response to a uniform flux that falls inside its 0.1° box and its
0.5° box (the rest is only on the 2.5° grid). Each curve is the share of soundings (y) for which
at most that fraction (x) lies in the box.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for column, label, colour in (("in_fine_box", "±5° box (0.1°)", FINE_COLOUR),
                              ("in_medium_box", "±25° box (0.5°)", MEDIUM_COLOUR)):  # fmt: skip
    values = np.sort(table[column].values)
    share = np.arange(1, len(values) + 1) / len(values)
    ax.step(values * 100, share * 100, where="post", color=colour, linewidth=2, label=label)
    median = np.median(values)
    ax.plot(
        median * 100,
        50,
        "o",
        color=colour,
        markersize=8,
        markeredgecolor="white",
        markeredgewidth=1.5,
        zorder=5,
    )
    ax.annotate(
        f"median {median:.0%}",
        (median * 100, 50),
        xytext=(-8, 6),
        textcoords="offset points",
        ha="right",
        fontsize=8.5,
        color=INK,
    )
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.set_xlabel("% of the sounding's response inside the box")
ax.set_ylabel("% of soundings")
ax.grid(color=FAINT, linewidth=0.5)
ax.set_axisbelow(True)
ax.legend(loc="upper left")
ax.set_title(f"Response inside each box, {len(table):,} soundings")
plt.show()

## Check: do the levels agree?
Finer cells should sum to the coarser cell they fill. For each sounding: the 0.1° box summed,
against the 0.5° cells it covers; and the 0.5° box summed, against the 2.5° cells it covers.
NAME converts to mixing ratio cell by cell, so they agree only approximately; in January 2013
the largest differences between the 0.5° and 2.5° levels (several %) are for soundings over the
Andes. The table lists the five largest.

In [ ]:
ratios = {"fine": [], "medium": []}
for sounding in table.index:
    coarse = sounding_footprint(ds, sounding, "coarse")
    medium = sounding_footprint(ds, sounding, "medium")
    fine = sounding_footprint(ds, sounding, "fine")
    on_medium = aggregate(fine, 5)
    covered = medium.sel(lat=on_medium["lat"], lon=on_medium["lon"], method="nearest")
    ratios["fine"].append(float(fine.sum() / covered.sum()))
    on_coarse = aggregate(medium, 5)
    covered = coarse.sel(lat=on_coarse["lat"], lon=on_coarse["lon"], method="nearest")
    ratios["medium"].append(float(medium.sum() / covered.sum()))
table["fine_over_medium"] = ratios["fine"]
table["medium_over_coarse"] = ratios["medium"]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, column, title, colour in (
    (axes[0], "fine_over_medium", "0.1° box vs the 0.5° cells it fills", FINE_COLOUR),
    (axes[1], "medium_over_coarse", "0.5° box vs the 2.5° cells it fills", MEDIUM_COLOUR),
):
    percent = (table[column] - 1) * 100
    ax.hist(percent, bins=np.linspace(-10, 10, 101), color=colour, edgecolor="white", linewidth=0.6)
    ax.axvline(0, color=INK, linewidth=0.8)
    ax.set_yscale("log")
    ax.set_xlabel("Difference of the box total (%)")
    ax.set_title(title)
    ax.grid(axis="y", color=FAINT, linewidth=0.5)
    ax.set_axisbelow(True)
    ax.text(
        0.98,
        0.95,
        f"median {percent.median():+.2f}%\n5–95%: {percent.quantile(0.05):+.2f} to"
        f" {percent.quantile(0.95):+.2f}%\nlargest |diff| {percent.abs().max():.1f}%",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=8,
        color=MUTED,
    )
axes[0].set_ylabel("Soundings")
plt.show()
table.loc[
    (table["medium_over_coarse"] - 1).abs().sort_values(ascending=False).index[:5],
    ["time", "lat", "lon", "medium_over_coarse", "fine_over_medium"],
]